# Temă de Programare: Compararea Strategiilor de Imputare

Bun venit la tema **Compararea Strategiilor de Imputare**!

Odata ce mecanismul de lipsa a fost caracterizat, urmatorul pas este alegerea si aplicarea unei strategii de imputare adecvate. Teoria acopera un spectru de optiuni:

| Strategie | Cand sa o folosesti |
|----------|----------------------|
| Listwise deletion | Datele sunt MCAR, lipsa < 5% |
| Mean imputation | Distributie simetrica, aproximativ normala, fara outlieri |
| Median imputation | Distributie asimetrica sau coloane cu outlieri |
| Mode imputation | Coloane categorice / ordinale |
| KNN imputation | MAR, relatii multivariate prezente |
| Iterative (MICE) | MAR, structura multivariata complexa |

**Ce vei face in aceasta tema:**

* Vei implementa complete-case analysis (listwise deletion)
* Vei aplica imputare cu media si mediana folosind `sklearn.impute.SimpleImputer`
* Vei trata coloanele categorice folosind imputare cu moda
* Vei folosi `KNNImputer` pentru completare sofisticata bazata pe vecini
* Vei aplica `IterativeImputer` (de tip MICE) pentru imputare iterativa multivariata
* Vei compara toate strategiile pe acelasi set de date intr-un singur raport

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI TALE:</h4>

* Toate celulele sunt blocate cu excepția celor în care trebuie să trimiți soluțiile sau atunci când este menționat explicit că poți interacționa cu ele.

* În fiecare celulă de exercițiu, caută comentariile `### ÎNCEPUT DE COD AICI ###` și `### SFÂRȘIT DE COD AICI ###`. Acestea îți arată unde să scrii codul soluției. **Nu adăuga sau modifica niciun cod care se află în afara acestor comentarii**.

* Poți adăuga celule noi pentru a experimenta, dar acestea vor fi omise de evaluator, deci nu te baza pe celule create nou pentru a găzdui codul soluției — folosește locurile prevăzute în acest scop.

---

## Cuprins
- [Importuri](#imports)
- [1 - Intelegerea Datelor](#1---understanding-the-data)
- [2 - Metode de Stergere](#2---deletion-methods)
    - **[Exercitiul 1 - `listwise_deletion`](#exercise-1---listwise_deletion)**
- [3 - Imputare Simpla](#3---simple-imputation)
    - **[Exercitiul 2 - `mean_median_imputation`](#exercise-2---mean_median_imputation)**
    - **[Exercitiul 3 - `mode_imputation`](#exercise-3---mode_imputation)**
- [4 - Imputare Avansata](#4---advanced-imputation)
    - **[Exercitiul 4 - `knn_imputation`](#exercise-4---knn_imputation)**
    - **[Exercitiul 5 - `iterative_imputation`](#exercise-5---iterative_imputation)**
- [5 - Compararea Strategiilor](#5---strategy-comparison)
    - **[Exercitiul 6 - `compare_imputation_strategies`](#exercise-6---compare_imputation_strategies)**


<a name='imports'></a>
## Importuri

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-the-data'></a>
## 1 - Intelegerea Datelor

Generam un set de date sintetic cu valori lipsa. Setul are valori ground-truth cunoscute, astfel incat calitatea imputarii poate fi masurata cu precizie.


In [ ]:
# Generate dataset — returns (df_missing, df_complete, missing_mask)
df_missing, df_complete, missing_mask = helper_utils.generate_dataset_with_missing(
    n_samples=200, missing_rate=0.15, random_state=42
)

print(f"Dataset shape: {df_missing.shape}")
print(f"\nMissing values per column:")
print(df_missing.isnull().sum())
print(f"\nTotal missing: {df_missing.isnull().sum().sum()}")

In [ ]:
helper_utils.visualize_missing_pattern(df_missing)

<a name='2---deletion-methods'></a>
## 2 - Metode de Stergere

Cea mai simpla abordare este **listwise deletion** (complete-case analysis): elimina fiecare rand care contine cel putin o valoare lipsa.

**Cand sa o folosesti:** Datele sunt MCAR si fractia de lipsa este mica (< 5%). In caz de MAR sau MNAR, eliminarea randurilor incomplete introduce bias, deoarece randurile lipsa sunt sistematic diferite de cele complete.


<a name='exercise-1---listwise_deletion'></a>
### **Exercitiul 1 - `listwise_deletion`**

**Sarcina ta:**

Implementeaza `listwise_deletion(df)` astfel incat sa elimine toate randurile care contin cel putin o valoare lipsa.

**Cerinte:**
- Returneaza un `pd.DataFrame` fara valori lipsa
- Nu modifica DataFrame-ul original

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

**Pasul 1:** `df.copy()` pentru a evita modificarea in-place

**Pasul 2:** `df.dropna()` elimina randurile cu *orice* valoare `NaN`

</details>


In [ ]:
# GRADED FUNCTION: listwise_deletion
def listwise_deletion(df):
    """
    Remove all rows containing any missing values.

    Args:
        df (pd.DataFrame): DataFrame with potential missing values.

    Returns:
        pd.DataFrame: DataFrame with no missing values.
    """
    ### ÎNCEPUT DE COD AICI ###

    result = None

    ### SFÂRȘIT DE COD AICI ###

    return result

In [ ]:
df_deleted = listwise_deletion(df_missing)
print(f"Original shape:  {df_missing.shape}")
print(f"After deletion:  {df_deleted.shape}")
print(f"Rows removed:    {len(df_missing) - len(df_deleted)}")
print(f"Missing remaining: {df_deleted.isnull().sum().sum()}")

In [ ]:
# Testează-ți codul!
unittests.exercise_1(listwise_deletion)

<a name='3---simple-imputation'></a>
## 3 - Imputare Simpla

Imputarea simpla inlocuieste fiecare valoare lipsa cu o singura statistica calculata. `sklearn.impute.SimpleImputer` suporta strategiile `'mean'`, `'median'`, `'most_frequent'` si `'constant'`.


<a name='exercise-2---mean_median_imputation'></a>
### **Exercitiul 2 - `mean_median_imputation`**

**Sarcina ta:**

Implementeaza `mean_median_imputation(df, strategy='mean')` astfel incat sa completeze valorile numerice lipsa cu media sau mediana coloanei.

**Cerinte:**
- Accepta `strategy` cu valorile `'mean'` sau `'median'`
- Foloseste `sklearn.impute.SimpleImputer`
- Returneaza un `pd.DataFrame` (nu un numpy array), cu numele coloanelor si indexul pastrate

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

**Pasul 1:** `imputer = SimpleImputer(strategy=strategy)`

**Pasul 2:** `imputed_array = imputer.fit_transform(df)`

**Pasul 3:** `pd.DataFrame(imputed_array, columns=df.columns, index=df.index)`

</details>


In [ ]:
# GRADED FUNCTION: mean_median_imputation
def mean_median_imputation(df, strategy='mean'):
    """
    Fill missing numeric values with column mean or median.

    Args:
        df (pd.DataFrame): DataFrame with missing values.
        strategy (str): 'mean' or 'median'.

    Returns:
        pd.DataFrame: DataFrame with imputed values, same shape as input.
    """
    ### ÎNCEPUT DE COD AICI ###

    imputer = None
    imputed_array = None
    result = None

    ### SFÂRȘIT DE COD AICI ###

    return result

In [ ]:
df_mean   = mean_median_imputation(df_missing, strategy='mean')
df_median = mean_median_imputation(df_missing, strategy='median')
print(f"After mean imputation   — missing: {df_mean.isnull().sum().sum()}")
print(f"After median imputation — missing: {df_median.isnull().sum().sum()}")

In [ ]:
# Testează-ți codul!
unittests.exercise_2(mean_median_imputation)

<a name='exercise-3---mode_imputation'></a>
### **Exercitiul 3 - `mode_imputation`**

**Sarcina ta:**

Implementeaza `mode_imputation(df, col)` astfel incat sa inlocuiasca valorile lipsa dintr-o coloana categorica cu cea mai frecventa valoare.

**Cerinte:**
- Completeaza doar valorile lipsa din coloana specificata `col`
- Foloseste `pd.Series.mode()[0]` pentru a gasi moda
- Returneaza DataFrame-ul modificat (copie)

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

**Pasul 1:** `result = df.copy()`

**Pasul 2:** `col_mode = df[col].mode()[0]`

**Pasul 3:** `result[col] = result[col].fillna(col_mode)`

</details>


In [ ]:
# GRADED FUNCTION: mode_imputation
def mode_imputation(df, col):
    """
    Replace missing values in a categorical column with the most frequent value.

    Args:
        df (pd.DataFrame): DataFrame with missing values.
        col (str): Column name to impute.

    Returns:
        pd.DataFrame: DataFrame with the specified column imputed.
    """
    ### ÎNCEPUT DE COD AICI ###

    result = None
    col_mode = None
    result[col] = None

    ### SFÂRȘIT DE COD AICI ###

    return result

In [ ]:
# Test on a sample categorical column
cat_df = pd.DataFrame({'category': ['A', 'B', 'A', np.nan, 'A', 'B', np.nan]})
result = mode_imputation(cat_df, 'category')
print("Before:", cat_df['category'].tolist())
print("After: ", result['category'].tolist())

In [ ]:
# Testează-ți codul!
unittests.exercise_3(mode_imputation)

<a name='4---advanced-imputation'></a>
## 4 - Imputare Avansata

### Imputare KNN

Din teorie: imputarea KNN gaseste cei $k$ cei mai similari *vecini completi* pentru fiecare rand care contine o valoare lipsa, apoi completeaza lipsa folosind media ponderata a valorilor vecinilor pentru acea coloana. Aceasta pastreaza mai bine structura locala a datelor decat imputarea simpla pe coloane.

**Compromisul principal:** Este costisitoare computational ($O(n^2)$ in implementarile naive) si sensibila la scala trasaturilor, deci standardizarea este necesara.

### Imputare Iterativa (MICE)

MICE trateaza fiecare coloana cu valori lipsa drept tinta de regresie. Fiecare coloana este imputata iterativ folosind toate celelalte coloane drept predictori pana la convergenta. Astfel este exploatata intreaga structura multivariata a setului de date. Este eficienta in regim MAR.


<a name='exercise-4---knn_imputation'></a>
### **Exercitiul 4 - `knn_imputation`**

**Sarcina ta:**

Implementeaza `knn_imputation(df, n_neighbors=5)` folosind `sklearn.impute.KNNImputer`.

**Cerinte:**
- Foloseste numarul de vecini specificat
- Returneaza un `pd.DataFrame` cu numele coloanelor si indexul pastrate

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

**Pasul 1:** `imputer = KNNImputer(n_neighbors=n_neighbors)`

**Pasul 2:** `imputed_array = imputer.fit_transform(df)`

**Pasul 3:** `pd.DataFrame(imputed_array, columns=df.columns, index=df.index)`

</details>


In [ ]:
# GRADED FUNCTION: knn_imputation
def knn_imputation(df, n_neighbors=5):
    """
    Fill missing values using K-Nearest Neighbors imputation.

    Args:
        df (pd.DataFrame): DataFrame with missing values (numeric columns only).
        n_neighbors (int): Number of neighbors to use.

    Returns:
        pd.DataFrame: DataFrame with imputed values.
    """
    ### ÎNCEPUT DE COD AICI ###

    imputer = None
    imputed_array = None
    result = None

    ### SFÂRȘIT DE COD AICI ###

    return result

In [ ]:
# Use only numeric columns for KNN
df_numeric = df_missing.select_dtypes(include='number')
df_knn = knn_imputation(df_numeric, n_neighbors=5)
print(f"After KNN imputation — missing: {df_knn.isnull().sum().sum()}")

In [ ]:
# Testează-ți codul!
unittests.exercise_4(knn_imputation)

<a name='exercise-5---iterative_imputation'></a>
### **Exercitiul 5 - `iterative_imputation`**

**Sarcina ta:**

Implementeaza `iterative_imputation(df, max_iter=10, random_state=42)` folosind `IterativeImputer` din sklearn.

**Cerinte:**
- Accepta `max_iter` pentru a controla ciclurile de convergenta
- Returneaza un `pd.DataFrame` cu numele coloanelor si indexul pastrate

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

**Pasul 1:** `imputer = IterativeImputer(max_iter=max_iter, random_state=random_state)`

**Pasul 2:** `imputed_array = imputer.fit_transform(df)`

**Pasul 3:** Converteste inapoi la `pd.DataFrame`, pastrand numele coloanelor si indexul original

</details>


In [ ]:
# GRADED FUNCTION: iterative_imputation
def iterative_imputation(df, max_iter=10, random_state=42):
    """
    Fill missing values using iterative (MICE-like) imputation.

    Args:
        df (pd.DataFrame): DataFrame with missing values (numeric columns only).
        max_iter (int): Maximum number of imputation cycles.
        random_state (int): Random seed for reproducibility.

    Returns:
        pd.DataFrame: DataFrame with imputed values.
    """
    ### ÎNCEPUT DE COD AICI ###

    imputer = None
    imputed_array = None
    result = None

    ### SFÂRȘIT DE COD AICI ###

    return result

In [ ]:
df_iterative = iterative_imputation(df_numeric, max_iter=10)
print(f"After iterative imputation — missing: {df_iterative.isnull().sum().sum()}")

In [ ]:
# Testează-ți codul!
unittests.exercise_5(iterative_imputation)

<a name='5---strategy-comparison'></a>
## 5 - Compararea Strategiilor

Acum ca ai implementat toate strategiile, aplica-le pe acelasi set de date si returneaza rezultatele intr-un dictionar pentru o comparatie usoara, una langa alta.


<a name='exercise-6---compare_imputation_strategies'></a>
### **Exercitiul 6 - `compare_imputation_strategies`**

**Sarcina ta:**

Implementeaza `compare_imputation_strategies(df)` astfel incat sa aplice toate cele trei strategii (mean, median, KNN) pe acelasi DataFrame numeric si sa returneze rezultatele intr-un dictionar.

**Cerinte:**
- DataFrame-ul de intrare trebuie sa contina doar coloane numerice
- Returneaza un `dict` cu cel putin cheile: `'mean'`, `'median'`, `'knn'`
- Fiecare valoare trebuie sa fie un `pd.DataFrame` fara valori lipsa

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

Aplica functiile pe care le-ai implementat anterior pe acelasi DataFrame de intrare.

```python
return {
    'mean':   mean_median_imputation(df, strategy='mean'),
    'median': mean_median_imputation(df, strategy='median'),
    'knn':    knn_imputation(df),
}
```

</details>


In [ ]:
# GRADED FUNCTION: compare_imputation_strategies
def compare_imputation_strategies(df):
    """
    Apply multiple imputation strategies to the same dataset and return results.

    Args:
        df (pd.DataFrame): Numeric DataFrame with missing values.

    Returns:
        dict: Keys are strategy names ('mean', 'median', 'knn'),
              values are imputed DataFrames with no missing values.
    """
    ### ÎNCEPUT DE COD AICI ###

    results = None

    ### SFÂRȘIT DE COD AICI ###

    return results

In [ ]:
comparison = compare_imputation_strategies(df_numeric)

print("Strategy Comparison — missing values remaining:")
for strategy, df_imp in comparison.items():
    print(f"  {strategy:10s}: {df_imp.isnull().sum().sum()} missing")

print("\nMean-imputed first 5 rows:")
comparison['mean'].head()

In [ ]:
# Testează-ți codul!
unittests.exercise_6(compare_imputation_strategies)

## 6 - Vizualizarea Diferentelor dintre Strategii

Ruleaza celula de mai jos pentru a compara vizual cum modifica fiecare strategie distributia unei trasaturi raportat la datele complete de tip ground-truth.


In [ ]:
helper_utils.compare_imputation_distributions(
    df_complete=df_complete.select_dtypes(include='number'),
    imputed_dict=comparison,
    missing_df=df_numeric
)